# H9: T1D Polygenic Risk Score and Spleen Architecture

**Aim:** Construct a simple unweighted T1D polygenic risk score (PRS) from 68 risk SNPs
and test whether cumulative genetic risk predicts spleen architecture better than
rs3184504 alone.

**Approach:** Unweighted PRS = sum(risk allele dosage) / (2 × N_genotyped), normalized
to [0,1]. Risk/Het/Protective annotations from Groups.xlsx are mapped to 0/1/2 dosage.

**Key limitation:** Only 13 samples have full 68-SNP panel (3 CODEX samples have only
rs3184504 or no SNP data). H9 is restricted to samples with ≥10 genotyped SNPs.

**Caveats:**
- Very small sample size for PRS (n=13 max, 10 PC-only)
- Unweighted PRS does not account for effect sizes
- Risk annotations may not reflect true effect directions for all SNPs
- CODEX vs Phenocycler platform confound

In [ ]:
import sys
sys.path.insert(0, '.')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import spearmanr

from data_utils import (
    load_all_data, build_feature_matrix, load_snp_panel, compute_prs,
    GENO_ORDER, GENO_PALETTE, CODEX_SAMPLES,
    setup_style, save_figure, save_table,
)

setup_style()

# Load data
df = load_all_data()
feat = build_feature_matrix(df)
project_samples = set(feat.index)

geno_df, dosage_df = load_snp_panel(project_samples)
prs = compute_prs(dosage_df)

# Merge PRS with feature matrix
merged = feat.reset_index().merge(prs.reset_index(), on='Sample', how='inner')
merged = merged.dropna(subset=['PRS'])
merged_pc = merged[merged['Platform'] == 'Phenocycler'].copy()

morph_cols = [c for c in feat.columns if c not in ('Genotype', 'Platform')]
top_morph = ['Follicle_density', 'Follicle_norm_density', 'RedPulp_density',
             'Follicle_count', 'Follicle_fraction', 'WP_fraction',
             'Vessel_median_area', 'Vessel_median_circularity']
top_morph = [c for c in top_morph if c in morph_cols]

print(f'Samples with PRS: {len(merged)} (All), {len(merged_pc)} (PC-only)')
print(f'SNPs in dosage matrix: {dosage_df.shape[1]} (polymorphic with risk annotation)')
print(f'PRS range: {merged["PRS"].min():.3f} - {merged["PRS"].max():.3f}')

## Section 1: SNP QC

In [ ]:
# Per-SNP metrics
snp_qc_rows = []
for snp in geno_df.columns:
    genos = geno_df[snp].dropna()
    n_genotyped = len(genos)
    unique = genos.unique()
    is_mono = len(unique) <= 1
    # In dosage matrix?
    in_dosage = snp in dosage_df.columns
    snp_qc_rows.append({
        'SNP': snp, 'N_genotyped': n_genotyped,
        'N_unique_genotypes': len(unique),
        'Monomorphic': is_mono,
        'In_dosage_matrix': in_dosage,
    })

snp_qc = pd.DataFrame(snp_qc_rows)
save_table(snp_qc, 'H9_snp_qc')

print(f'Total SNPs: {len(snp_qc)}')
print(f'Polymorphic with risk annotation: {snp_qc["In_dosage_matrix"].sum()}')
print(f'Monomorphic: {snp_qc["Monomorphic"].sum()}')
print(f'\nPer-sample genotyping completeness:')
sample_completeness = dosage_df.notna().sum(axis=1)
for sid in sorted(sample_completeness.index):
    print(f'  {sid}: {sample_completeness[sid]}/{dosage_df.shape[1]} SNPs')

In [ ]:
# Genotype dosage heatmap (samples x SNPs)
plot_dosage = dosage_df.loc[dosage_df.index.isin(project_samples)].copy()
# Sort samples by genotype
sample_order = []
for geno in GENO_ORDER:
    geno_samples = [s for s in plot_dosage.index
                    if feat.loc[s, 'Genotype'] == geno] if True else []
    try:
        geno_samples = sorted(s for s in plot_dosage.index
                              if s in feat.index and str(feat.loc[s, 'Genotype']) == geno)
    except Exception:
        pass
    sample_order.extend(geno_samples)

if sample_order:
    plot_dosage = plot_dosage.loc[[s for s in sample_order if s in plot_dosage.index]]

fig, ax = plt.subplots(figsize=(18, 6))
cmap = plt.cm.YlOrRd.copy()
cmap.set_bad('lightgray')
sns.heatmap(plot_dosage, cmap=cmap, vmin=0, vmax=2, ax=ax,
            cbar_kws={'label': 'Risk dosage (0=Prot, 1=Het, 2=Risk)', 'shrink': 0.6},
            xticklabels=True, yticklabels=True)
ax.set_xlabel('SNP')
ax.set_ylabel('Sample')
ax.set_title('Genotype Risk Dosage Heatmap')
plt.xticks(fontsize=6, rotation=90)
plt.yticks(fontsize=9)
fig.tight_layout()
save_figure(fig, 'H9_snp_qc_heatmap')
plt.show()

## Section 2: PRS Construction and Distribution

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

# Histogram
ax = axes[0]
ax.hist(merged['PRS'], bins=8, edgecolor='black', color='steelblue', alpha=0.7)
ax.set_xlabel('PRS')
ax.set_ylabel('Count')
ax.set_title(f'PRS Distribution (n={len(merged)})')

# Strip by genotype
ax = axes[1]
sns.stripplot(data=merged, x='Genotype', y='PRS', order=GENO_ORDER,
              palette=GENO_PALETTE, size=10, edgecolor='black', linewidth=0.5, ax=ax)
sns.boxplot(data=merged, x='Genotype', y='PRS', order=GENO_ORDER,
            palette=GENO_PALETTE, ax=ax, linewidth=0.8, fliersize=0,
            boxprops=dict(alpha=0.3))
ax.set_title('PRS by rs3184504 Genotype')

# PRS vs rs3184504 dosage
ax = axes[2]
merged['rs3184504_dosage'] = merged['Genotype'].astype(str).map({'C/C': 0, 'C/T': 1, 'T/T': 2})
r_prs_geno, p_prs_geno = spearmanr(merged['rs3184504_dosage'], merged['PRS'])
ax.scatter(merged['rs3184504_dosage'], merged['PRS'],
           c=[GENO_PALETTE[g] for g in merged['Genotype']],
           s=80, edgecolors='black', linewidths=0.5)
ax.set_xticks([0, 1, 2])
ax.set_xticklabels(GENO_ORDER)
ax.set_xlabel('rs3184504 Dosage')
ax.set_ylabel('PRS')
ax.set_title(f'PRS vs rs3184504 (\u03c1={r_prs_geno:.2f}, p={p_prs_geno:.3f})')

fig.suptitle('Polygenic Risk Score Construction', fontsize=14, y=1.02)
fig.tight_layout()
save_figure(fig, 'H9_prs_distribution')
plt.show()

## Section 3: PRS vs Morphology

In [ ]:
# Spearman: PRS vs features, compared to rs3184504 dosage
comparison_rows = []
for mvar in top_morph:
    valid_prs = merged[['PRS', mvar]].dropna()
    valid_dosage = merged[['rs3184504_dosage', mvar]].dropna()
    r_prs, p_prs = spearmanr(valid_prs['PRS'], valid_prs[mvar]) if len(valid_prs) >= 4 else (np.nan, np.nan)
    r_dos, p_dos = spearmanr(valid_dosage['rs3184504_dosage'], valid_dosage[mvar]) if len(valid_dosage) >= 4 else (np.nan, np.nan)
    comparison_rows.append({
        'Feature': mvar,
        'PRS_rho': r_prs, 'PRS_p': p_prs,
        'rs3184504_rho': r_dos, 'rs3184504_p': p_dos,
        'n': len(valid_prs)
    })

comparison_df = pd.DataFrame(comparison_rows)
display(comparison_df)

# Paired bar chart
fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(len(comparison_df))
width = 0.35
bars1 = ax.bar(x - width/2, comparison_df['PRS_rho'].abs(), width,
               label='PRS (68 SNPs)', color='coral', edgecolor='black', linewidth=0.5)
bars2 = ax.bar(x + width/2, comparison_df['rs3184504_rho'].abs(), width,
               label='rs3184504 only', color='steelblue', edgecolor='black', linewidth=0.5)

# Significance markers
for i, row in comparison_df.iterrows():
    for bars, p_col, offset in [(bars1, 'PRS_p', -width/2), (bars2, 'rs3184504_p', width/2)]:
        p = row[p_col]
        if p < 0.05:
            ax.text(i + offset, bars[i].get_height() + 0.02, '*', ha='center', fontsize=14)

ax.set_xticks(x)
ax.set_xticklabels(comparison_df['Feature'], rotation=45, ha='right')
ax.set_ylabel('|Spearman \u03c1|')
ax.set_title('PRS vs rs3184504-only: Correlation with Morphological Features')
ax.legend()
ax.axhline(0, color='gray', linewidth=0.5)
fig.tight_layout()
save_figure(fig, 'H9_prs_comparison')
plt.show()

In [ ]:
# 2x3 scatter: PRS vs top 6 features
n_scatter = min(6, len(top_morph))
nrows = (n_scatter + 2) // 3
fig, axes = plt.subplots(nrows, 3, figsize=(15, 4.5 * nrows), squeeze=False)

for idx, mvar in enumerate(top_morph[:n_scatter]):
    ax = axes[idx // 3, idx % 3]
    for geno in GENO_ORDER:
        sub = merged[merged['Genotype'] == geno]
        ax.scatter(sub['PRS'], sub[mvar], c=[GENO_PALETTE[geno]],
                   label=geno, s=60, edgecolors='black', linewidths=0.5)
    valid = merged[['PRS', mvar]].dropna()
    if len(valid) >= 4:
        r, p = spearmanr(valid['PRS'], valid[mvar])
        ax.set_title(f'{mvar}\n\u03c1={r:.2f}, p={p:.3f}', fontsize=10)
    else:
        ax.set_title(mvar, fontsize=10)
    ax.set_xlabel('PRS')
    ax.set_ylabel(mvar)
    if idx == 0:
        ax.legend(fontsize=8)

# Hide unused
for j in range(n_scatter, nrows * 3):
    axes[j // 3, j % 3].set_visible(False)

fig.suptitle('PRS vs Morphological Features', fontsize=14, y=1.02)
fig.tight_layout()
save_figure(fig, 'H9_prs_scatter')
plt.show()

## Section 4: SNP Component Analysis

In [ ]:
# Find the feature most correlated with PRS
best_feat = comparison_df.loc[comparison_df['PRS_rho'].abs().idxmax(), 'Feature']
print(f'Top PRS-correlated feature: {best_feat}')

# Per-SNP |rho| with that feature
snp_contribs = []
for snp in dosage_df.columns:
    snp_vals = dosage_df[snp].loc[dosage_df.index.isin(merged['Sample'])]
    feat_vals = merged.set_index('Sample').loc[snp_vals.index, best_feat]
    valid = pd.DataFrame({'snp': snp_vals, 'feat': feat_vals}).dropna()
    if len(valid) >= 4 and valid['snp'].nunique() > 1:
        r, p = spearmanr(valid['snp'], valid['feat'])
        snp_contribs.append({'SNP': snp, 'rho': r, 'abs_rho': abs(r), 'p': p})

snp_contrib_df = pd.DataFrame(snp_contribs).sort_values('abs_rho', ascending=True)

# Bar chart
fig, ax = plt.subplots(figsize=(8, max(6, len(snp_contrib_df) * 0.3)))
colors = ['red' if s == 'rs3184504' else 'steelblue' for s in snp_contrib_df['SNP']]
ax.barh(snp_contrib_df['SNP'], snp_contrib_df['abs_rho'], color=colors,
        edgecolor='black', linewidth=0.3)
ax.set_xlabel(f'|Spearman \u03c1| with {best_feat}')
ax.set_title(f'Per-SNP Correlation with {best_feat}\n(rs3184504 highlighted in red)')
ax.axvline(0, color='gray', linewidth=0.5)
fig.tight_layout()
save_figure(fig, 'H9_snp_contributions')
plt.show()

## Section 5: Platform Sensitivity

In [ ]:
print(f'PC-only samples with PRS: {len(merged_pc)}')

if len(merged_pc) >= 4:
    pc_comparison = []
    for mvar in top_morph:
        valid_all = merged[['PRS', mvar]].dropna()
        valid_pc = merged_pc[['PRS', mvar]].dropna()
        r_all, p_all = spearmanr(valid_all['PRS'], valid_all[mvar]) if len(valid_all) >= 4 else (np.nan, np.nan)
        r_pc, p_pc = spearmanr(valid_pc['PRS'], valid_pc[mvar]) if len(valid_pc) >= 4 else (np.nan, np.nan)
        pc_comparison.append({
            'Feature': mvar,
            'All_rho': r_all, 'All_p': p_all, 'All_n': len(valid_all),
            'PC_rho': r_pc, 'PC_p': p_pc, 'PC_n': len(valid_pc),
        })

    pc_comp_df = pd.DataFrame(pc_comparison)
    display(pc_comp_df)

    # Concordance at p<0.1
    concordant = sum(
        1 for _, r in pc_comp_df.iterrows()
        if (r['All_p'] < 0.1) == (r['PC_p'] < 0.1)
    )
    print(f'\nConcordance (p<0.1 threshold): {concordant}/{len(pc_comp_df)}')
else:
    print('Too few PC-only samples for platform sensitivity analysis.')

## Summary Tables

In [ ]:
# PRS values
prs_out = merged[['Sample', 'Genotype', 'Platform', 'PRS']].copy()
prs_out['N_SNPs_genotyped'] = prs_out['Sample'].map(
    dosage_df.notna().sum(axis=1).to_dict())
save_table(prs_out, 'H9_prs_values')

# Statistical tests
stats_rows = []
for _, row in comparison_df.iterrows():
    stats_rows.append({
        'Test': 'Spearman (PRS)', 'Feature': row['Feature'],
        'Statistic': row['PRS_rho'], 'p': row['PRS_p'], 'n': row['n']
    })
    stats_rows.append({
        'Test': 'Spearman (rs3184504)', 'Feature': row['Feature'],
        'Statistic': row['rs3184504_rho'], 'p': row['rs3184504_p'], 'n': row['n']
    })
stats_rows.append({
    'Test': 'PRS-rs3184504 correlation', 'Feature': 'PRS vs dosage',
    'Statistic': r_prs_geno, 'p': p_prs_geno, 'n': len(merged)
})

stats_df = pd.DataFrame(stats_rows)
save_table(stats_df, 'H9_statistical_tests')
print(f'Saved PRS values ({len(prs_out)} samples) and {len(stats_df)} statistical tests.')

In [ ]:
# Key findings summary
print('=' * 60)
print('H9: Polygenic Risk Score \u2014 Key Findings')
print('=' * 60)

print(f'\nSamples with PRS: {len(merged)} (min_snps=10)')
print(f'Polymorphic risk-annotated SNPs: {dosage_df.shape[1]}')
print(f'PRS range: {merged["PRS"].min():.3f} \u2013 {merged["PRS"].max():.3f}')
print(f'PRS-rs3184504 dosage correlation: \u03c1={r_prs_geno:.3f}, p={p_prs_geno:.3f}')

print('\nPRS vs morphology (top by |rho|):')
for _, r in comparison_df.nsmallest(5, 'PRS_p').iterrows():
    better = 'PRS' if abs(r['PRS_rho']) > abs(r['rs3184504_rho']) else 'rs3184504'
    print(f'  {r["Feature"]}: PRS \u03c1={r["PRS_rho"]:.3f} (p={r["PRS_p"]:.3f}) '
          f'vs rs3184504 \u03c1={r["rs3184504_rho"]:.3f} (p={r["rs3184504_p"]:.3f}) '
          f'\u2192 {better} stronger')